In [1]:
import pandas as pd
import json
video_df = pd.read_csv("../Dataset/QK-video_subset_5M.csv")
article_df = pd.read_csv("../Dataset/QK-article_subset_5M_preprocessed.csv")

# Tính mean cho numeric features
video_defaults = video_df[['watching_times']].mean().to_dict()
article_defaults = article_df[['exposure_count','click_count','like_count','comment_count',
                               'read_percentage','item_score1','item_score2','item_score3','read_time']].mean().to_dict()

# Gộp vào 1 dictionary lớn
numeric_defaults = {
    "video": video_defaults,
    "article": article_defaults
}

# ✅ In ra màn hình để kiểm tra
print(json.dumps(numeric_defaults, indent=4))

# ✅ Lưu ra file JSON
with open("../src-backend/numeric_defaults.json", "w") as f:  
    json.dump(numeric_defaults, f, indent=4)

print("✅ Đã lưu numeric_defaults.json")

{
    "video": {
        "watching_times": 1.2347368
    },
    "article": {
        "exposure_count": 2.7392843549023383e-17,
        "click_count": 2.4304824819409988e-17,
        "like_count": 8.969891496235504e-18,
        "comment_count": -9.14326392376097e-18,
        "read_percentage": -9.78275238594506e-18,
        "item_score1": 8.19355250314402e-17,
        "item_score2": 1.685009465290932e-16,
        "item_score3": 1.2403233995428308e-16,
        "read_time": 2.7196733753953595e-17
    }
}
✅ Đã lưu numeric_defaults.json


In [2]:
import pandas as pd

# Đọc dữ liệu
file_path = "../Dataset/QK-article_subset_5M_preprocessed.csv"
df = pd.read_csv(file_path)

# Xem phân phối của cột 'read'
read_counts = df['read'].value_counts(normalize=True)
print("📊 Phân phối class (tỉ lệ %):")
print(read_counts)

# Xem tổng số dòng theo class
print("\n🔢 Tổng số dòng:")
print(df['read'].value_counts())

📊 Phân phối class (tỉ lệ %):
read
1    0.97308
0    0.02692
Name: proportion, dtype: float64

🔢 Tổng số dòng:
read
1    4865400
0     134600
Name: count, dtype: int64


In [3]:
# Lấy tất cả dòng có read = 0
df_0 = df[df['read'] == 0]

# Lấy ngẫu nhiên số dòng tương ứng từ read = 1
df_1_sampled = df[df['read'] == 1].sample(n=len(df_0), random_state=42)

# Gộp lại và shuffle
df_balanced = pd.concat([df_0, df_1_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

# Kiểm tra lại phân phối
print("✅ Dataset sau khi balance:")
print(df_balanced['read'].value_counts(normalize=True))


✅ Dataset sau khi balance:
read
0    0.5
1    0.5
Name: proportion, dtype: float64


In [4]:
# Lưu dataset sau khi xử lý để train
output_path = "../Dataset/QK-article_subset_balanced.csv"
df_balanced.to_csv(output_path, index=False)
print(f"📁 Dataset đã lưu tại: {output_path}")

📁 Dataset đã lưu tại: ../Dataset/QK-article_subset_balanced.csv


In [5]:
# Xem thử 5 dòng đầu
df_balanced.head()

,user_id,item_id,gender,age,exposure_count,click_count,like_count,comment_count,read_percentage,item_score1,item_score2,category_second,category_first,item_score3,read,read_time,share,like,follow,favorite
0,4282016,8605,2,1,-0.092038,0.608580,4.425550,0.361694,-0.582551,-0.937996,-0.079590,294,34,-0.407320,0,-0.655668,0,0,0,0
1,4185754,8535,2,6,10.766747,4.132374,-0.425469,-0.484889,-1.309720,-3.443419,-4.713336,0,0,-2.402666,0,-0.655668,0,0,0,0
2,4168306,12845,2,1,-0.095109,0.018384,0.770439,-0.337501,-1.025176,0.314715,-0.079590,319,41,0.590353,1,-0.539541,0,0,0,0
3,4184419,8535,2,1,10.766747,4.132374,-0.425469,-0.484889,-1.309720,-3.443419,-4.713336,0,0,-2.402666,0,-0.655668,0,0,0,0
4,4295861,8535,1,1,10.766747,4.132374,-0.425469,-0.484889,-1.309720,-3.443419,-4.713336,0,0,-2.402666,0,-0.655668,0,0,0,0


In [6]:
import pickle

# Đường dẫn đến file feature_index cũ (LabelEncoder)
old_path = "Deep FM/article/feature_index.pkl"
# Đường dẫn lưu file mới (dict)
new_path = "Deep FM/article/feature_index_dict.pkl"

with open(old_path, "rb") as f:
    feature_index = pickle.load(f)

converted_index = {}

for col, encoder in feature_index.items():
    if hasattr(encoder, "classes_"):
        # LabelEncoder: lấy value → index
        converted_index[col] = {val: idx for idx, val in enumerate(encoder.classes_)}
    else:
        # Nếu đã là dict rồi thì giữ nguyên
        converted_index[col] = encoder

# Lưu file mới
with open(new_path, "wb") as f:
    pickle.dump(converted_index, f)

print("✅ Đã chuyển đổi feature_index về dạng dict và lưu tại:", new_path)


✅ Đã chuyển đổi feature_index về dạng dict và lưu tại: Deep FM/article/feature_index_dict.pkl
